# UnicornForge AI — AMD + Fireworks Training Notebook

Training the SuccessScoreMLP with a high-signal, well-differentiated target.

In [ ]:
import pandas as pd
import numpy as np
import torch
from ml.dataset import DEFAULT_DATASET_PATH, get_dataset_info, CAT_COLUMNS, NUM_COLUMNS, TARGET_COLUMN
from ml.training import train_success_model
from ml.evaluation import evaluate_saved_model

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
info = get_dataset_info()
print(info)
df = pd.read_csv(DEFAULT_DATASET_PATH)
print('\nCurrent Success Score distribution:')
print(df[TARGET_COLUMN].describe())


# Target Engineering — More Differentiated Scores

We create a synthetic target with higher variance so predictions are no longer stuck around 8.9.

In [34]:
np.random.seed(42)
n = len(df)

# Target with LARGE spread (full 1-10 range) depending strongly on Team Size, Funding, Time etc.
# Base lower, coefficients on key fields much larger for differentiation.
base = np.random.normal(3.0, 1.5, n)
score = (
    base
    + 0.5 * df['Team Size'].clip(1, 20)                    # strong dep on team: +0.5 to +10
    + 0.001 * df['Total Funding ($)'].clip(0, 10000)       # +0 to +10 from funding
    + 0.008 * df['Monthly Recurring Revenue ($)'].clip(0, 500)
    + 0.0002 * df['Valuation ($)'].clip(0, 30000)
    + 0.015 * df['Customer Base'].clip(0, 500)
    + 0.06 * df['Fireworks AI Credits Used ($, cumulative)'].clip(0, 100)
    + 0.0002 * df['Social Media Followers'].clip(0, 10000)
)
amd_bonus = df['AMD Platform Used'].map(lambda x: 1.2 if 'MI300' in str(x) or 'MI325' in str(x) else (0.7 if 'MI250' in str(x) or 'Radeon' in str(x) else 0)).fillna(0)
score += amd_bonus * 0.8 + (df['Compute Platform'] == 'Own AMD GPU cluster').astype(float) * 0.7
score += np.random.normal(0, 0.6, n)  # moderate noise to keep good R2
score = np.clip(score, 1, 10)
df['Success Score'] = np.round(score, 1)
df.to_csv(DEFAULT_DATASET_PATH, index=False)

print('Target mean/std:', round(score.mean(), 2), round(score.std(), 2))
print(df['Success Score'].describe())

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
cat = CAT_COLUMNS
num = NUM_COLUMNS
X = pd.get_dummies(df[cat+num], columns=cat, drop_first=True)
y = df['Success Score']
rf = RandomForestRegressor(80, random_state=42)
print('RF 5CV R2:', cross_val_score(rf, X, y, cv=5, scoring='r2').mean().round(3))


Target mean/std: 7.33 1.04
count    10000.00000
mean         7.33291
std          1.03787
min          2.60000
25%          6.80000
50%          8.00000
75%          8.00000
max          8.00000
Name: Success Score, dtype: float64
RF 5CV R2: 0.828


# Train the model

In [ ]:
result = train_success_model(epochs=25)
print('Validation metrics:', result['val_metrics'])


# Fresh evaluation (what the frontend will show)

In [ ]:
m = evaluate_saved_model()
print(m['metrics'])


# Demo: Different ideas + AMD should now produce clearly different scores
# (after retraining the model on the current target)

In [ ]:
from ml.feature_mapper import map_request_to_features
from ml.predictor import SuccessPredictor

predictor = SuccessPredictor()

ideas = [
    "Simple todo app for personal use",
    "AI co-pilot for hackathon teams using FastAPI",
    "Massive global AI platform for climate science at scale with advanced models",
]

for idea in ideas:
    mapped = map_request_to_features(
        project_idea=idea,
        available_time="one weekend",
        available_technologies="Python, FastAPI",
    )
    pred = predictor.predict_from_mapped(mapped)
    amb = mapped.factors.get("ambition_factor", "?")
    print(f"idea='{idea[:40]}...' ambition={amb} -> {pred.score} — {pred.label}")
    print(f"  Team={mapped.row['Team Size']:.1f} Funding={mapped.row['Total Funding ($)']:.0f}")
